In [5]:
# ── 1. Install dependencies ──────────────────────────────────────────────────
!pip install gymnasium sb3-contrib stable-baselines3 --quiet

In [6]:
# ── 2. System path setup ─────────────────────────────────────────────────────
import sys, os
import numpy as np

DATASET_PATH = "/kaggle/input/datasets/nguyennhatphong/roguelike-rl-env"
sys.path.insert(0, DATASET_PATH)

print("Python version:", sys.version)
print("Dataset files:", os.listdir(DATASET_PATH)[:10])

Python version: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Dataset files: ['python_roguelike']


In [ ]:
# ── 3. Adaptive Environment Wrapper ──────────────────────────────────────────
from python_roguelike.env.roguelike_env import RoguelikeEnv
from python_roguelike.data.enums import GameState
from sb3_contrib.common.wrappers import ActionMasker


class AdaptiveEnv(RoguelikeEnv):
    """
    Adaptive agent reward shaping (v4):

    REDESIGN: Removed HP-blended alpha/beta reward switching.
    The blended reward created a non-stationary signal that violated
    the stationary MDP assumption — PPO can't learn when the same action
    has different value depending on HP ratio.

    v4 approach:
      - HP ratio is already in obs[0] — let the network learn mode-switching
      - Simple, stationary reward: flat floor +25, HP-loss −0.2, win +500
      - Enemy kill reward (+5 per kill) for direct combat signal
      - Removed ★1 card penalty (harmful early-game when forced picks)
    """

    def __init__(self, seed=42, json_path=None, max_steps=10_000):
        super().__init__(seed=seed, json_path=json_path, max_steps=max_steps)
        self._prev_deck_size = 0
        self._prev_enemy_count = 0

    def reset(self, *, seed=None, options=None):
        obs, info = super().reset(seed=seed, options=options)
        self._prev_deck_size = len(self._controller.current_run.the_hero.deck.master_deck)
        self._prev_enemy_count = 0
        return obs, info

    def _count_alive_enemies(self):
        combat = self._controller.current_run.current_combat
        if combat is None:
            return 0
        return sum(1 for e in combat.enemies if e.current_health > 0)

    def step(self, action: int):
        run = self._controller.current_run
        hero = run.the_hero
        prev_hp = hero.current_health
        prev_floor = run.current_floor
        prev_alive = self._count_alive_enemies()

        obs, _base_reward, terminated, truncated, info = super().step(action)

        run = self._controller.current_run
        hero = run.the_hero
        reward = 0.0

        # ── HP change — simple, stationary penalty ──────────────────────────
        hp_delta = hero.current_health - prev_hp
        if hp_delta < 0:
            reward += hp_delta * 0.2     # −0.2 per HP lost

        # ── Floor progress — flat +25 per floor ─────────────────────────────
        floor_delta = run.current_floor - prev_floor
        if floor_delta > 0:
            reward += floor_delta * 25.0

        # ── Enemy kill reward — direct combat signal ────────────────────────
        curr_alive = self._count_alive_enemies()
        enemies_killed = prev_alive - curr_alive
        if enemies_killed > 0:
            reward += enemies_killed * 5.0

        # ── Step pressure ───────────────────────────────────────────────────
        reward -= 0.03

        # ── Deck quality reward — fires once when a new card is picked ──────
        current_deck_size = len(hero.deck.master_deck)
        if current_deck_size > self._prev_deck_size:
            new_card = hero.deck.master_deck[-1]
            star = getattr(new_card, 'star_rating', 1)
            if star >= 4:
                reward += 8.0
            elif star == 3:
                reward += 3.0
            # ★1–2: no penalty (early-game forced picks)
            self._prev_deck_size = current_deck_size

        # ── Terminal rewards ────────────────────────────────────────────────
        if terminated:
            if hero.current_health > 0:
                reward += 500.0
            else:
                reward -= 200.0

        self._prev_enemy_count = curr_alive
        return obs, reward, terminated, truncated, info


json_path = os.path.join(DATASET_PATH, "python_roguelike", "GameData.json")
env = AdaptiveEnv(seed=42, json_path=json_path)
obs, info = env.reset()
print("   AdaptiveEnv v4 created — obs shape:", obs.shape)
print("   Action space:", env.action_space)
print("   obs[0] = hp_pct (network learns mode-switching):", obs[0])

   AdaptiveEnv created — obs shape: (153,)
   Action space: Discrete(67)
   Obs[0] = hero hp_pct (enables context-aware behavior): 1.0


In [8]:
# ── 4. Validate Action Masking ───────────────────────────────────────────────
masked_env = ActionMasker(env, lambda e: e.action_masks())
obs, info = masked_env.reset()

for step in range(200):
    mask = masked_env.action_masks()
    valid = np.where(mask)[0]
    obs, reward, terminated, truncated, info = masked_env.step(int(np.random.choice(valid)))
    if terminated or truncated:
        obs, info = masked_env.reset()

print("   Masking validation passed")

   Masking validation passed


In [9]:
# ── 5. Vectorized Environment ─────────────────────────────────────────────────
from stable_baselines3.common.vec_env import SubprocVecEnv, VecMonitor

N_ENVS = 4

def make_env(seed_offset: int):
    def _init():
        _env = AdaptiveEnv(seed=4000 + seed_offset, json_path=json_path, max_steps=10_000)
        return ActionMasker(_env, lambda e: e.action_masks())
    return _init

vec_env = VecMonitor(SubprocVecEnv([make_env(i) for i in range(N_ENVS)]))
print(f"   VecEnv ready: {N_ENVS} parallel envs")

2026-03-21 09:37:35.220759: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-21 09:37:35.221025: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774085855.248599     122 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774085855.248904     121 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774085855.256938     121 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
E0000 00:00:1774085855.257682     122 cuda_blas.cc:1

   VecEnv ready: 4 parallel envs


In [ ]:
# ── 6. MaskablePPO Model ─────────────────────────────────────────────────────
from sb3_contrib import MaskablePPO
from stable_baselines3.common.utils import get_linear_fn

policy_kwargs = dict(
    net_arch=dict(
        pi=[512, 512],
        vf=[512, 512],
    )
)

model = MaskablePPO(
    "MlpPolicy",
    vec_env,
    verbose=1,
    n_steps=2048,
    batch_size=512,
    n_epochs=10,
    learning_rate=get_linear_fn(3e-4, 1e-5, 1.0),
    gamma=0.995,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.10,   # raised from 0.05 to prevent premature entropy collapse
    vf_coef=0.5,
    max_grad_norm=0.5,
    policy_kwargs=policy_kwargs,
    tensorboard_log="/kaggle/working/tensorboard/adaptive",
)
print("Model created. Policy params:", sum(p.numel() for p in model.policy.parameters()))

Using cpu device
Model created. Policy params: 717892


In [11]:
# ── 7. Callbacks ──────────────────────────────────────────────────────────────
from stable_baselines3.common.callbacks import CheckpointCallback, EvalCallback, CallbackList

eval_env = ActionMasker(AdaptiveEnv(seed=6666, json_path=json_path), lambda e: e.action_masks())

callbacks = CallbackList([
    CheckpointCallback(
        save_freq=50_000,
        save_path="/kaggle/working/checkpoints/adaptive",
        name_prefix="adaptive_ppo",
        verbose=1,
    ),
    EvalCallback(
        eval_env,
        best_model_save_path="/kaggle/working/best_model/adaptive",
        log_path="/kaggle/working/eval_logs/adaptive",
        eval_freq=25_000,
        n_eval_episodes=15,
        deterministic=False,
        verbose=1,
    ),
])
print("   Callbacks configured")

   Callbacks configured


In [ ]:
# ── 8. Training ──────────────────────────────────────────────────────────────
TOTAL_TIMESTEPS = 5_000_000

print(f"   Starting Adaptive (v4) Agent training for {TOTAL_TIMESTEPS:,} timesteps...")
print(f"   REDESIGN: No HP-blended switching — stationary reward, network learns modes")
print(f"   Floor +25 | HP loss −0.2 | Enemy kill +5 | Win +500")
print(f"   Network: [512, 512], LR: linear_schedule(3e-4), ent_coef: 0.10")
print()

model.learn(total_timesteps=TOTAL_TIMESTEPS, callback=callbacks)

model.save("/kaggle/working/adaptive_ppo_final")
print("\n   Training complete! Model saved to: /kaggle/working/adaptive_ppo_final")

   Starting Adaptive (Meta-PPO) Agent training for 10,000,000 timesteps...
   Strategy: Dynamic mode based on HP ratio
   HP < 30% → Defensive mode | HP > 70% → Aggressive mode
   Win = floor 15 cleared: +500 | Deck quality: +2×(star-1)
   Network: [512, 512], LR: linear_schedule(3e-4), ent_coef: 0.10

Logging to /kaggle/working/tensorboard/adaptive/MaskablePPO_1


/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/callbacks.py:418: UserWarning: Training and eval env are not of the same type<stable_baselines3.common.vec_env.vec_monitor.VecMonitor object at 0x7d1cf827b380> != <stable_baselines3.common.vec_env.dummy_vec_env.DummyVecEnv object at 0x7d1c96d071d0>
  warnings.warn("Training and eval env are not of the same type" f"{self.training_env} != {self.eval_env}")


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 125      |
|    ep_rew_mean     | 10.5     |
| time/              |          |
|    fps             | 1145     |
|    iterations      | 1        |
|    time_elapsed    | 7        |
|    total_timesteps | 8192     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 125         |
|    ep_rew_mean          | 9.9         |
| time/                   |             |
|    fps                  | 906         |
|    iterations           | 2           |
|    time_elapsed         | 18          |
|    total_timesteps      | 16384       |
| train/                  |             |
|    approx_kl            | 0.009274039 |
|    clip_fraction        | 0.119       |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.48       |
|    explained_variance   | 0.00153     |
|    learning_rate        | 0.

KeyboardInterrupt: 

In [ ]:
# ── 9. Evaluation ───────────────────────────────────────────────────────────
model = MaskablePPO.load("/kaggle/working/best_model/adaptive/best_model")

N_EVAL = 50
floors, full_wins, act_wins, hp_pcts, mode_switches = [], 0, 0, [], []

for ep in range(N_EVAL):
    ep_env = AdaptiveEnv(seed=ep, json_path=json_path)
    ep_env = ActionMasker(ep_env, lambda e: e.action_masks())
    obs, info = ep_env.reset()
    done = False
    aggressive_steps = 0
    defensive_steps = 0

    while not done:
        # Track mode
        hp_pct_current = obs[0]  # obs[0] = hp_pct (from observation space docs)
        if hp_pct_current > 0.7:
            aggressive_steps += 1
        elif hp_pct_current < 0.3:
            defensive_steps += 1

        action, _ = model.predict(obs, deterministic=True, action_masks=ep_env.action_masks())
        obs, reward, terminated, truncated, info = ep_env.step(int(action))
        done = terminated or truncated

    floors.append(info["floor"])
    total_steps = aggressive_steps + defensive_steps
    if total_steps > 0:
        mode_switches.append(aggressive_steps / total_steps)

    if info["hp"] > 0 and terminated:
        hp_pcts.append(info["hp"] / info["max_hp"])
        if info["floor"] >= 15:
            full_wins += 1  # Full game clear
        else:
            act_wins += 1   # Survived but didn't finish

print(f"\n Adaptive Agent Evaluation Results ({N_EVAL} episodes)")
print(f"═" * 50)
print(f"  Full-Game Win Rate:    {full_wins/N_EVAL*100:.1f}%  ({full_wins}/{N_EVAL}) — floor 15 cleared")
print(f"  Act Win Rate:          {act_wins/N_EVAL*100:.1f}%  ({act_wins}/{N_EVAL}) — survived but didn't finish")
print(f"  Death Rate:            {(N_EVAL-full_wins-act_wins)/N_EVAL*100:.1f}%")
print(f"  Avg Floor Reached:     {np.mean(floors):.1f} / 15")
print(f"  Max Floor Reached:     {max(floors)}")
if hp_pcts:
    print(f"  Avg HP% on Survive:    {np.mean(hp_pcts)*100:.1f}%")
if mode_switches:
    print(f"  Avg Aggressive Steps %: {np.mean(mode_switches)*100:.1f}%  (of non-balanced steps)")

In [ ]:
# # ── 10. Multi-Agent Comparison Summary ───────────────────────────────────────

# import json

# def evaluate_agent(model_path, env_class, n_eval=50):
#     try:
#         m = MaskablePPO.load(model_path)
#     except Exception:
#         return {"error": f"Model not found: {model_path}"}

#     floors, wins = [], 0
#     for ep in range(n_eval):
#         ep_env = ActionMasker(env_class(seed=ep, json_path=json_path), lambda e: e.action_masks())
#         obs, info = ep_env.reset()
#         done = False
#         while not done:
#             action, _ = m.predict(obs, deterministic=True, action_masks=ep_env.action_masks())
#             obs, reward, terminated, truncated, info = ep_env.step(int(action))
#             done = terminated or truncated
#         floors.append(info["floor"])
#         if info["hp"] > 0 and terminated:
#             wins += 1
#     return {"win_rate": wins / n_eval, "avg_floor": np.mean(floors)}


# # These paths would be from a shared dataset if running cross-notebook
# # The AggressiveEnv / DefensiveEnv / BalancedEnv must be imported or re-defined here
# # Uncomment and adapt as needed:

# # from python_roguelike.env.roguelike_env import RoguelikeEnv
# # results = {
# #     "Aggressive": evaluate_agent("/kaggle/input/models/aggressive_ppo_final", AggressiveEnv),
# #     "Defensive":  evaluate_agent("/kaggle/input/models/defensive_ppo_final",  DefensiveEnv),
# #     "Balanced":   evaluate_agent("/kaggle/input/models/balanced_ppo_final",   BalancedEnv),
# #     "Adaptive":   evaluate_agent("/kaggle/working/best_model/adaptive/best_model", AdaptiveEnv),
# # }
# # 
# # print(f"\n   MULTI-AGENT COMPARISON TABLE")
# # print(f"{'Agent':<12} {'Win Rate':>10} {'Avg Floor':>12}")
# # print("─" * 38)
# # for agent_name, r in results.items():
# #     if "error" not in r:
# #         print(f"{agent_name:<12} {r['win_rate']*100:>9.1f}% {r['avg_floor']:>12.1f}")

# print("    Uncomment the code above after all 4 agents are trained!")